# SAP Kanban Analyzer - Testovací Notebook

Tento notebook ti umožní testovat jednotlivé komponenty aplikace a interaktivně analyzovat data.

## Před spuštěním:
1. Ujisti se, že SAP GUI je spuštěná a jsi přihlášen
2. Povolené SAP GUI Scripting (Options → Accessibility & Scripting)
3. Vyplň config.yaml se svými údaji

## 1. Import knihoven a inicializace

In [ ]:
import pandas as pd
import numpy as np
import yaml
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

# Import našich modulů
from sap_connection import SAPConnection
from material_analyzer import MaterialAnalyzer
from kanban_evaluator import KanbanEvaluator

# Nastavení pro pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Nastavení pro grafy
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Knihovny načteny")

## 2. Načtení konfigurace

In [ ]:
# Načti config
with open('config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print("Konfigurace:")
print(f"  Werk: {config['analysis']['plant']}")
print(f"  Typy materiálů: {config['analysis']['material_types']}")
print(f"  Období: {config['analysis']['analysis_period_months']} měsíců")
print(f"\nKanban kritéria:")
for key, value in config['analysis']['kanban_criteria'].items():
    print(f"  {key}: {value}")

## 3. Připojení k SAP

In [ ]:
# Vytvoř připojení
sap = SAPConnection()

# Připoj se
if sap.connect():
    print("✓ Připojeno k SAP")
    print(f"  Systém: {sap.session.Info.SystemName}")
    print(f"  Klient: {sap.session.Info.Client}")
    print(f"  Uživatel: {sap.session.Info.User}")
else:
    print("❌ Nepodařilo se připojit k SAP")
    print("Ujisti se, že SAP GUI je spuštěná a jsi přihlášen")

## 4. Test SAP připojení - jednoduchý příkaz

In [ ]:
# Zkus spustit transakci SE38 jako test
if sap.is_connected():
    sap.start_transaction("SE38")
    print("✓ Transakce SE38 spuštěna")
    
    # Vrať se zpět
    sap.send_vkey(3)  # F3 = Back
    print("✓ Test dokončen")

## 5. Získání dat zásob (MB52)

In [ ]:
# Inicializuj analyzer
analyzer = MaterialAnalyzer(sap)

# Získej data zásob
plant = config['analysis']['plant']
material_type = config['analysis']['material_types'][0]  # První typ

print(f"Získávám zásoby pro werk {plant}, typ {material_type}...")
print("(Toto může trvat několik minut podle množství dat)")

stock_data = analyzer.get_materials_stock(plant, material_type)

if not stock_data.empty:
    print(f"\n✓ Získáno {len(stock_data)} materiálů")
    print(f"\nSloupce: {list(stock_data.columns)}")
    print("\nPrvních 5 řádků:")
    display(stock_data.head())
else:
    print("❌ Žádná data")

## 6. Získání pohybů materiálů (MB51)

In [ ]:
# Nastav období
months = config['analysis']['analysis_period_months']
date_from = datetime.now() - timedelta(days=months * 30)
date_to = datetime.now()

print(f"Získávám pohyby od {date_from.date()} do {date_to.date()}...")
print("(Toto může trvat delší dobu při velkém množství pohybů)")

movements_data = analyzer.get_material_movements(
    plant=plant,
    material="*",  # Všechny materiály
    date_from=date_from,
    date_to=date_to
)

if not movements_data.empty:
    print(f"\n✓ Získáno {len(movements_data)} pohybů")
    print(f"\nSloupce: {list(movements_data.columns)}")
    print("\nPrvních 10 řádků:")
    display(movements_data.head(10))
    
    # Základní statistiky
    print("\nStatistiky pohybů:")
    print(movements_data.describe())
else:
    print("❌ Žádná data pohybů")

## 7. Analýza spotřeby

In [ ]:
# POZNÁMKA: Tady musíš upravit názvy sloupců podle toho, co vrátí SAP
# Níže jsou příklady typických názvů sloupců

print("Analyzuji spotřebu...")

# Zkontroluj dostupné sloupce
print(f"Dostupné sloupce v movements_data: {list(movements_data.columns)}")
print("\nUprav názvy sloupců v následující buňce podle tvých dat!")

# Příklad - uprav tyto názvy!
consumption_analysis = analyzer.analyze_material_consumption(
    movements_data,
    material_column="Material",  # ← UPRAV
    quantity_column="Quantity",   # ← UPRAV
    date_column="Posting Date",   # ← UPRAV
    movement_type_column="Movement Type"  # ← UPRAV
)

if not consumption_analysis.empty:
    print(f"\n✓ Analyzováno {len(consumption_analysis)} materiálů")
    print("\nPrvních 10 materiálů:")
    display(consumption_analysis.head(10))
else:
    print("❌ Analýza selhala")

## 8. Vyhodnocení Kanban kandidátů

In [ ]:
# Vytvoř evaluátor
evaluator = KanbanEvaluator(config['analysis']['kanban_criteria'])

# Vyhodnoť kandidáty
print("Vyhodnocuji Kanban kandidáty...")
results = evaluator.evaluate_kanban_candidates(
    consumption_analysis,
    stock_data
)

if not results.empty:
    print(f"\n✓ Vyhodnoceno {len(results)} materiálů")
    
    # Doporučené materiály
    recommended = results[results['Kanban_Recommended'] == True]
    print(f"\nDoporučeno pro Kanban: {len(recommended)} materiálů")
    
    # Zobraz top 10
    print("\nTop 10 kandidátů:")
    display(recommended.head(10)[[
        'Material', 'Kanban_Score', 'Movements_Per_Month',
        'Consumption_Regularity', 'Potential_Savings_Movements',
        'Recommendation_Reasons'
    ]])
else:
    print("❌ Vyhodnocení selhalo")

## 9. Vizualizace výsledků

In [ ]:
# Graf 1: Distribuce Kanban skóre
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Histogram skóre
axes[0, 0].hist(results['Kanban_Score'], bins=20, edgecolor='black')
axes[0, 0].axvline(0.6, color='red', linestyle='--', label='Práh doporučení')
axes[0, 0].set_xlabel('Kanban Score')
axes[0, 0].set_ylabel('Počet materiálů')
axes[0, 0].set_title('Distribuce Kanban skóre')
axes[0, 0].legend()

# Scatter: Frekvence vs Pravidelnost
scatter = axes[0, 1].scatter(
    results['Movements_Per_Month'],
    results['Consumption_Regularity'],
    c=results['Kanban_Score'],
    cmap='RdYlGn',
    alpha=0.6
)
axes[0, 1].set_xlabel('Vyskladnění za měsíc')
axes[0, 1].set_ylabel('Pravidelnost spotřeby')
axes[0, 1].set_title('Frekvence vs Pravidelnost')
plt.colorbar(scatter, ax=axes[0, 1], label='Kanban Score')

# Top 10 úspor
top_savings = recommended.nlargest(10, 'Potential_Savings_Movements')
axes[1, 0].barh(
    range(len(top_savings)),
    top_savings['Potential_Savings_Movements']
)
axes[1, 0].set_yticks(range(len(top_savings)))
axes[1, 0].set_yticklabels(top_savings['Material'])
axes[1, 0].set_xlabel('Úspora vyskladnění/rok')
axes[1, 0].set_title('Top 10 materiálů - potenciální úspory')
axes[1, 0].invert_yaxis()

# Pie chart: Doporučené vs Nedoporučené
kanban_counts = results['Kanban_Recommended'].value_counts()
axes[1, 1].pie(
    kanban_counts,
    labels=['Nedoporučeno', 'Doporučeno pro Kanban'],
    autopct='%1.1f%%',
    colors=['#ff9999', '#90ee90']
)
axes[1, 1].set_title('Podíl kandidátů pro Kanban')

plt.tight_layout()
plt.show()

print("✓ Grafy vygenerovány")

## 10. Summary Report

In [ ]:
# Vytvoř summary
summary = evaluator.generate_summary_report(results)

print("="*60)
print("SUMMARY REPORT")
print("="*60)
print(f"\nCelkem materiálů: {summary['total_materials_analyzed']}")
print(f"Doporučeno pro Kanban: {summary['total_kanban_candidates']} ({summary['recommendation_rate']})")
print(f"Potenciální úspora: {summary['total_potential_savings_movements_per_year']} vyskladnění/rok")
print(f"Průměrné skóre: {summary['average_kanban_score']}")

print("\n" + "="*60)
print("TOP 10 KANDIDÁTŮ")
print("="*60)

for i, candidate in enumerate(summary['top_10_candidates'], 1):
    print(f"\n{i}. {candidate['Material']}")
    print(f"   Score: {candidate['Kanban_Score']:.2f}")
    print(f"   Vyskladnění/měsíc: {candidate['Movements_Per_Month']:.1f}")
    print(f"   Pravidelnost: {candidate['Consumption_Regularity']:.0%}")
    print(f"   Úspora: ~{candidate['Potential_Savings_Movements']} vyskladnění/rok")

## 11. Export do Excel

In [ ]:
# Export
output_file = config['output']['excel_report']

print(f"Exportuji do {output_file}...")
evaluator.export_to_excel(results, summary, output_file)
print(f"✓ Export dokončen")
print(f"\nSoubor obsahuje:")
print("  - All Materials: Všechny analyzované materiály")
print("  - Recommended for Kanban: Jen doporučené materiály")
print("  - Summary: Souhrnná statistika")
print("  - Top 10 Candidates: Top 10 kandidátů")

## 12. Odpojení od SAP

In [ ]:
# Odpoj se
sap.disconnect()
print("✓ Odpojeno od SAP")

---

## Další analýzy

Tady můžeš přidat vlastní analýzy a dotazy na data:

In [ ]:
# Příklad: Filtruj materiály s vysokou frekvencí ale nízkou pravidelností
high_freq_low_reg = results[
    (results['Movements_Per_Month'] > 20) & 
    (results['Consumption_Regularity'] < 0.5)
]

print(f"Materiály s vysokou frekvencí ale nízkou pravidelností: {len(high_freq_low_reg)}")
display(high_freq_low_reg[[
    'Material', 'Movements_Per_Month', 'Consumption_Regularity', 
    'Kanban_Score', 'Kanban_Recommended'
]])

In [ ]:
# Příklad: Materiály těsně pod prahem - potenciální kandidáti s úpravou kritérií
near_threshold = results[
    (results['Kanban_Score'] >= 0.5) & 
    (results['Kanban_Score'] < 0.6)
]

print(f"Materiály těsně pod prahem: {len(near_threshold)}")
display(near_threshold.head(10)[[
    'Material', 'Kanban_Score', 'Movements_Per_Month',
    'Consumption_Regularity', 'Recommendation_Reasons'
]])